## Price movement vs. result vs. performance

Core question: does the Kalshi market move more with the *actual result* (goal margin) or with *underlying performance* (xG margin, shots, possession)? And does it reprice gradually during play or jump at the final whistle?

For each of the 104 matches, the winning-outcome market's price series is split into two pieces:

- **`in_match_movement`** — `|price 5 min before close - price at kickoff|`. Repricing that happened *during* play, while the outcome was still genuinely uncertain.
- **`settlement_jump`** — `|1.0 - price 5 min before close|`. The discontinuous jump to the settled price right at the end — i.e. how much the market *hadn't* already priced in before the final whistle.
- **`price_range`** — total movement (max close - min close) across the whole series, for comparison with `dominance_analysis.ipynb`.

All three are regressed on `goal_margin` (the result) and `xg_margin` (the performance signal), standardized to compare effect sizes directly.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.formula.api as smf

# Notebook lives in code/, so data/ and output/ live one level up.
ROOT = Path.cwd().parent
FIG_DIR = ROOT / "output" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

TEAM_NAME_ALIASES = {
    "Cape Verde": "Cabo Verde",
    "Bosnia and Herzegovina": "Bosnia & Herzegovina",
    "Congo DR": "DR Congo",
    "Ivory Coast": "C\u00f4te d'Ivoire",
    "IR Iran": "Iran",
    "Turkiye": "T\u00fcrkiye",
    "Curacao": "Cura\u00e7ao",
    "Korea Republic": "South Korea",
}

### Kalshi: split each match's price series into in-match movement vs. settlement jump

In [ ]:
def load_price_dynamics() -> pd.DataFrame:
    markets = pd.read_parquet(ROOT / "data/kalshi/kxwcgame_markets.parquet")
    candles = pd.read_parquet(ROOT / "data/kalshi/candlesticks/kxwcgame_minute.parquet")
    schedule = pd.read_parquet(ROOT / "data/sofascore/schedule.parquet")

    outcome_markets = markets[markets["result"] == "yes"].copy()
    parts = outcome_markets["event_title"].str.split(" vs ", n=1, expand=True)
    team1 = parts[0].str.strip().replace(TEAM_NAME_ALIASES)
    team2 = parts[1].str.split(":").str[0].str.strip().replace(TEAM_NAME_ALIASES)
    outcome_markets["team_set"] = [frozenset(t) for t in zip(team1, team2)]

    sched = schedule.copy()
    sched["team_set"] = [frozenset(t) for t in zip(sched["home_team"], sched["away_team"])]
    outcome_markets = outcome_markets.merge(
        sched[["team_set", "start_time"]], on="team_set", how="left"
    )

    candles = candles.merge(
        outcome_markets[["market_ticker", "start_time", "close_time"]], on="market_ticker"
    )
    candles["close_time"] = pd.to_datetime(candles["close_time"])

    rows = []
    for ticker, grp in candles.groupby("market_ticker"):
        grp = grp.sort_values("timestamp")
        kickoff, close = grp["start_time"].iloc[0], grp["close_time"].iloc[0]

        at_kickoff = grp[grp["timestamp"] >= kickoff]
        price_at_kickoff = (
            at_kickoff["price_close"].iloc[0] if len(at_kickoff) else grp["price_close"].iloc[0]
        )

        # "5 min before close" as a proxy for the last pre-settlement belief,
        # just ahead of the discontinuous jump to the settled 0/1 price.
        pre_settle = grp[grp["timestamp"] <= close - pd.Timedelta(minutes=5)]
        price_pre_settlement = (
            pre_settle["price_close"].iloc[-1] if len(pre_settle) else grp["price_close"].iloc[-1]
        )

        rows.append(
            {
                "market_ticker": ticker,
                "price_range": grp["price_close"].max() - grp["price_close"].min(),
                "in_match_movement": abs(price_pre_settlement - price_at_kickoff),
                "settlement_jump": abs(1.0 - price_pre_settlement),
            }
        )

    dynamics = outcome_markets.merge(pd.DataFrame(rows), on="market_ticker")
    return dynamics[["team_set", "price_range", "in_match_movement", "settlement_jump"]]

### SofaScore + result: performance margins and the actual goal margin

In [ ]:
def load_performance_features() -> pd.DataFrame:
    stats = pd.read_parquet(ROOT / "data/sofascore/statistics.parquet")
    schedule = pd.read_parquet(ROOT / "data/sofascore/schedule.parquet")
    stats_all = stats[stats["period"] == "ALL"]

    def margin(stat_name: str, colname: str) -> pd.DataFrame:
        # Some stats (e.g. "Total shots") are listed under more than one
        # SofaScore group_name ("Match overview" AND "Shots") with identical
        # values -- dedupe per (event_id, stat_name) or every merge below
        # silently doubles rows for every match that has the duplicate.
        d = stats_all[stats_all["stat_name"] == stat_name].drop_duplicates(subset=["event_id"])
        d = d.merge(schedule[["event_id", "home_team", "away_team"]], on="event_id")
        d["team_set"] = [frozenset(t) for t in zip(d["home_team"], d["away_team"])]
        d[colname] = (d["home_value"] - d["away_value"]).abs()
        return d[["team_set", colname]]

    perf = (
        margin("Expected goals", "xg_margin")
        .merge(margin("Ball possession", "possession_margin"), on="team_set")
        .merge(margin("Total shots", "shots_margin"), on="team_set")
    )

    sched = schedule.copy()
    sched["team_set"] = [frozenset(t) for t in zip(sched["home_team"], sched["away_team"])]
    sched["goal_margin"] = (sched["home_score"] - sched["away_score"]).abs()

    return perf.merge(sched[["team_set", "goal_margin"]], on="team_set")

In [ ]:
dynamics = load_price_dynamics()
performance = load_performance_features()

df = dynamics.merge(performance, on="team_set", how="inner")
print(f"{len(df)} matches with both price dynamics and performance data (of {len(dynamics)} total)")

# Standardize predictors so regression coefficients are directly
# comparable as "effect per std dev", regardless of each stat's own units
# (goals vs. xG vs. possession %).
for col in ["goal_margin", "xg_margin", "possession_margin", "shots_margin"]:
    df[f"{col}_z"] = (df[col] - df[col].mean()) / df[col].std()

df[["price_range", "in_match_movement", "settlement_jump", "goal_margin", "xg_margin"]].describe()

### Regressions

**A — total price movement**: result vs. performance, head to head.
**B — in-match movement only**: does price track performance *during* play?
**C — the settlement jump**: how much does the market get caught off guard at the final whistle?

In [ ]:
model_a = smf.ols("price_range ~ goal_margin_z + xg_margin_z", data=df).fit()
print(model_a.summary())

In [ ]:
model_b = smf.ols(
    "in_match_movement ~ xg_margin_z + possession_margin_z + shots_margin_z", data=df
).fit()
print(model_b.summary())

In [ ]:
model_c = smf.ols("settlement_jump ~ goal_margin_z + xg_margin_z", data=df).fit()
print(model_c.summary())

### Comparing standardized effect sizes across models

One coefficient plot, all three models side by side: point = coefficient (change in the outcome per 1 std dev of the predictor), whiskers = 95% CI. A whisker crossing zero means that predictor isn't a reliable driver of that outcome once the other predictor(s) are controlled for.

In [ ]:
rows = []
for model_name, model in [("A: price_range", model_a), ("B: in_match_movement", model_b), ("C: settlement_jump", model_c)]:
    ci = model.conf_int()
    for predictor in model.params.index:
        if predictor == "Intercept":
            continue
        rows.append(
            {
                "model": model_name,
                "predictor": predictor.replace("_z", "").replace("_", " "),
                "coef": model.params[predictor],
                "ci_low": ci.loc[predictor, 0],
                "ci_high": ci.loc[predictor, 1],
            }
        )
coef_df = pd.DataFrame(rows)

models = coef_df["model"].unique()
fig, axes = plt.subplots(1, len(models), figsize=(13, 4), sharey=False)
for ax, model_name in zip(axes, models):
    sub = coef_df[coef_df["model"] == model_name].iloc[::-1]
    y = range(len(sub))
    err = [sub["coef"] - sub["ci_low"], sub["ci_high"] - sub["coef"]]
    colors = ["tab:red" if lo <= 0 <= hi else "tab:blue" for lo, hi in zip(sub["ci_low"], sub["ci_high"])]
    ax.errorbar(sub["coef"], y, xerr=err, fmt="o", capsize=3, color="black", ecolor="gray")
    ax.scatter(sub["coef"], y, color=colors, zorder=3, s=60)
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_yticks(list(y))
    ax.set_yticklabels(sub["predictor"])
    ax.set_title(model_name, fontsize=10)
    ax.set_xlabel("standardized coefficient")

fig.suptitle("What moves the Kalshi price: result vs. performance (95% CI; red crosses zero)")
fig.tight_layout()
out_path = FIG_DIR / "regression_coefficients.png"
fig.savefig(out_path, dpi=150)
print(f"Saved {out_path}")